# 🌐 Cross-Channel Subscription Clustering & LLM-Driven Market Segmentation
### *Uncovering Cross-Audience Media Affinity Clusters, Video Performance Mining, and Conceptual Refinement via Gemini 3.1 Flash-Lite*

---


## 📌 Executive Summary & Analytical Methodology

### Abstract
This Jupyter Notebook implements a multi-stage machine learning and LLM synthesis pipeline to segment cross-channel audience subscription profiles into meaningful media consumption clusters. Building upon the Top 100 comment author subscriptions extracted in `youtube_comment_author_subscriptions.ipynb`, this workflow filters for channels with at least more than 1 occurrence ($f(c) > 1$) among the target audience. For each qualified channel, the notebook queries the **YouTube Data API v3** (`playlistItems.list` on long-form `UULF` playlist IDs) to retrieve metadata for the most recent 50 long-form videos ($>60$ seconds duration). Textual embeddings are computed across video titles, averaged per channel, and partitioned into **8 distinct clusters** using K-Means clustering. Subsequently, an LLM-driven qualitative synthesis engine (**Gemini 3.1 Flash-Lite** via `google-genai` SDK) processes the clusters sequentially from largest to smallest—incorporating previous cluster descriptions as context to prevent generic output overlap. Finally, a meta-synthesis LLM request evaluates all cluster descriptions holistically to refine conceptual clusters beyond numerical boundaries, exporting dual-payload reports (.md and .json) locally and directly to **Google Drive** (`/content/drive/MyDrive/YouTube_Analytics/`).

### Key Analytical Pipeline Stages:
1. **Subscription Import & Occurrence Filtering**: Imports the aggregated subscription dataset and isolates channels subscribed to by at least 2 or more unique comment authors ($f(c) > 1$).
2. **Quota-Efficient Long-Form Video Mining**: Resolves `UULF` playlist IDs (filtering out Shorts $\le 60$s via ISO 8601 duration checks) to collect up to 50 recent long-form videos per channel. Exports this consolidated dataset directly to Google Drive.
3. **Channel Embedding Vectorization & K-Means Clustering**: Generates text embeddings for video titles using Gemini API / TF-IDF feature vectors, computes mean channel embedding vectors $v_c = \frac{1}{N_c} \sum_{i=1}^{N_c} e(t_{c,i})$, and groups channels into 8 clusters ($K=8$).
4. **Sequential LLM Cluster Description Synthesis**: Orders clusters from largest to smallest by channel count. Calls `gemini-3.1-flash-lite` iteratively, injecting previously generated cluster descriptions as context to ensure distinct, non-generic descriptions (including Title, Short Description, Lengthy Explanation with examples, and Top 10 performing videos per channel).
5. **Holistic LLM Refinement & Dual-Payload Persistence**: Submits all initial cluster descriptions to a final LLM request to synthesize refined, higher-level conceptual clusters. Exports a Markdown report (`{Channel_Name}_refined_subscription_clusters.md`) containing the refined cluster descriptions followed by individual cluster breakdowns (with channel titles and top 3 performing videos per channel).


## ⚙️ 1. Configuration & Setup

In this cell, we initialize global user parameters, API credentials, model specifications (`gemini-3.1-flash-lite`), cluster target count ($K=8$), and output storage directories. Credentials are first read from Google Colab Secrets (`YOUTUBE_API_KEY`, `GOOGLE_API_KEY`) with fallbacks to environment variables.

In [ ]:
# Install required packages if missing
!pip install -q google-api-python-client google-genai pandas matplotlib seaborn scikit-learn numpy

import os
import re
import time
import json
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from googleapiclient.discovery import build
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer

# Try loading API keys from Colab Secrets if available
yt_api_key = ""
google_api_key = ""
try:
    from google.colab import userdata
    yt_api_key = userdata.get('YOUTUBE_API_KEY') or userdata.get('GOOGLE_API_KEY')
    google_api_key = userdata.get('GOOGLE_API_KEY') or userdata.get('YOUTUBE_API_KEY')
except Exception:
    pass

# User Configuration
CONFIG = {
    "YOUTUBE_API_KEY": yt_api_key or os.environ.get("YOUTUBE_API_KEY", "YOUR_YOUTUBE_API_KEY_HERE"),
    "GOOGLE_API_KEY": google_api_key or os.environ.get("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE"),
    "MODEL_NAME": "gemini-3.1-flash-lite",
    "EMBEDDING_MODEL": "text-embedding-004",
    "CHANNEL_IDENTIFIER": "@josemillanastrologohumanista",
    "MIN_SUBSCRIPTION_OCCURRENCES": 2,      # Only consider channels with > 1 occurrence
    "TARGET_VIDEOS_PER_CHANNEL": 50,        # Recent long-form videos to fetch per channel
    "NUM_CLUSTERS": 8,                       # Target 8 clusters for K-Means
    "DRIVE_OUTPUT_DIR": "/content/drive/MyDrive/YouTube_Analytics",
    "LOCAL_FALLBACK_DIRS": ["data", "analysis", "."],
    "CACHE_FILE_NAME": "llm_generation_cache.json",
    "MAX_RETRIES": 3,
    "BACKOFF_FACTOR": 2,
    "MOUNT_DRIVE": True,
    "USE_CACHE": True
}

print("Configuration initialized successfully.")
print(f"Target Model: {CONFIG['MODEL_NAME']} | Target Clusters: {CONFIG['NUM_CLUSTERS']}")

## 💾 2. Mount Google Drive & Storage Hierarchy

Mounts Google Drive at `/content/drive/MyDrive/YouTube_Analytics/` with a robust local fallback hierarchy (`/content/drive/MyDrive/YouTube_Analytics`, `data`, `analysis`, `.`) to guarantee seamless execution across Google Colab and local environments.

In [ ]:
ACTIVE_OUTPUT_DIR = "."

if CONFIG["MOUNT_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.makedirs(CONFIG["DRIVE_OUTPUT_DIR"], exist_ok=True)
        ACTIVE_OUTPUT_DIR = CONFIG["DRIVE_OUTPUT_DIR"]
        print(f"✅ Google Drive mounted successfully. Primary path: {ACTIVE_OUTPUT_DIR}")
    except ImportError:
        print("ℹ️ google.colab not available. Operating in local environment.")
    except Exception as e:
        print(f"⚠️ Mount failed ({e}). Falling back to local hierarchy.")

if ACTIVE_OUTPUT_DIR == ".":
    for fallback in ["/content/drive/MyDrive/YouTube_Analytics", "data", "analysis", "."]:
        if os.path.exists(fallback) or fallback in [".", "data", "analysis"]:
            os.makedirs(fallback, exist_ok=True)
            ACTIVE_OUTPUT_DIR = fallback
            break

print(f"📁 Active storage path: {os.path.abspath(ACTIVE_OUTPUT_DIR)}")

## 🔍 3. YouTube API Client, Helper Utilities & Persistent LLM Caching

Defines utility functions for YouTube Data API v3 client initialization, ISO 8601 duration parsing (>60s long-form isolation), filename sanitization, and persistent LLM caching with a 3-retry exponential backoff mechanism.

In [ ]:
def get_youtube_client(api_key: str):
    if not api_key or api_key == "YOUR_YOUTUBE_API_KEY_HERE":
        return None
    return build('youtube', 'v3', developerKey=api_key)

def parse_iso8601_duration(duration_str: str) -> int:
    pattern = re.compile(r'PT(?:(?P<hours>\d+)H)?(?:(?P<minutes>\d+)M)?(?:(?P<seconds>\d+)S)?')
    match = pattern.match(duration_str or "")
    if not match:
        return 0
    parts = match.groupdict()
    hours = int(parts['hours']) if parts['hours'] else 0
    minutes = int(parts['minutes']) if parts['minutes'] else 0
    seconds = int(parts['seconds']) if parts['seconds'] else 0
    return hours * 3600 + minutes * 60 + seconds

def sanitize_filename(name: str) -> str:
    return re.sub(r'[^a-zA-Z0-9_-]', '_', name).strip('_')

class PersistentLLMCache:
    def __init__(self, primary_dir: str, fallback_dirs: list, filename: str):
        self.filename = filename
        self.search_paths = [primary_dir] + fallback_dirs
        self.cache: dict = self._load_cache()

    def _load_cache(self) -> dict:
        for d in self.search_paths:
            p = os.path.join(d, self.filename)
            if os.path.exists(p):
                try:
                    with open(p, "r", encoding="utf-8") as f:
                        return json.load(f)
                except Exception:
                    pass
        return {}

    def get(self, key: str):
        return self.cache.get(key)

    def set(self, key: str, value: str):
        self.cache[key] = value
        for d in self.search_paths:
            try:
                os.makedirs(d, exist_ok=True)
                with open(os.path.join(d, self.filename), "w", encoding="utf-8") as f:
                    json.dump(self.cache, f, indent=2, ensure_ascii=False)
            except Exception:
                pass

llm_cache = PersistentLLMCache(ACTIVE_OUTPUT_DIR, CONFIG["LOCAL_FALLBACK_DIRS"], CONFIG["CACHE_FILE_NAME"])

def call_gemini_with_retry(prompt: str, cache_key: str = None) -> str:
    if CONFIG["USE_CACHE"] and cache_key:
        cached = llm_cache.get(cache_key)
        if cached:
            print(f"⚡ Reusing cached LLM response for key: {cache_key[:40]}...")
            return cached

    api_key = CONFIG["GOOGLE_API_KEY"]
    if not api_key or api_key == "YOUR_GOOGLE_API_KEY_HERE":
        raise RuntimeError("Cannot call Gemini API: GOOGLE_API_KEY is missing.")

    try:
        from google import genai
        from google.genai import types
    except ImportError:
        raise ImportError("google-genai SDK not installed. Please run pip install google-genai.")

    client = genai.Client(api_key=api_key)
    max_retries = CONFIG["MAX_RETRIES"]
    backoff = CONFIG["BACKOFF_FACTOR"]
    last_err = None

    for attempt in range(1, max_retries + 1):
        try:
            res = client.models.generate_content(
                model=CONFIG["MODEL_NAME"],
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.3, top_p=0.95)
            )
            out_text = res.text
            if out_text:
                if CONFIG["USE_CACHE"] and cache_key:
                    llm_cache.set(cache_key, out_text)
                return out_text
            raise ValueError("Empty response payload received from Gemini.")
        except Exception as e:
            last_err = e
            print(f"⚠️ Attempt {attempt} failed: {e}")
            if attempt < max_retries:
                time.sleep(backoff * attempt)

    raise RuntimeError(f"Exhausted all {max_retries} retries calling Gemini API. Last error: {last_err}")

print("Utility functions and persistent caching engine initialized.")

## 📂 4. Import Exported Subscriptions & Filter Channels (>1 Occurrence)

Loads the top subscribed channels dataset exported by `3.Youtube_comment_author_subscriptions.ipynb` across Google Drive and local search directories. Strictly filters for channels that have **at least more than one occurrence** (`author_count > 1` or `total_occurrences > 1`).

In [ ]:
safe_channel_id = sanitize_filename(CONFIG["CHANNEL_IDENTIFIER"])
search_dirs = [ACTIVE_OUTPUT_DIR, "/content/drive/MyDrive/YouTube_Analytics", "data", "analysis", "."]

top_channels_data = []
found_sub_file = None

for d in search_dirs:
    if not os.path.exists(d):
        continue
    for fname in os.listdir(d):
        if "top_100_subscribed_channels" in fname or "subscriptions" in fname:
            if fname.endswith(".json"):
                fpath = os.path.join(d, fname)
                try:
                    with open(fpath, "r", encoding="utf-8") as f:
                        payload = json.load(f)
                    if isinstance(payload, dict) and "top_channels" in payload:
                        top_channels_data = payload["top_channels"]
                        found_sub_file = fpath
                        break
                    elif isinstance(payload, list):
                        top_channels_data = payload
                        found_sub_file = fpath
                        break
                except Exception as e:
                    print(f"⚠️ Error reading candidate subscription file {fpath}: {e}")
    if found_sub_file:
        break

df_top_channels_raw = pd.DataFrame(top_channels_data) if top_channels_data else pd.DataFrame()

# Filter for channels with > 1 occurrence
if not df_top_channels_raw.empty:
    # Check occurrence column candidates
    cnt_col = "author_count" if "author_count" in df_top_channels_raw.columns else "total_occurrences"
    df_filtered_channels = df_top_channels_raw[df_top_channels_raw[cnt_col] >= CONFIG["MIN_SUBSCRIPTION_OCCURRENCES"]].copy()
else:
    df_filtered_channels = pd.DataFrame()

print(f"☑ Subscription Dataset Import Metrics:")
print(f"   - Source Subscription File:           {found_sub_file}")
print(f"   - Total Channels in Imported List:    {len(df_top_channels_raw)}")
print(f"   - Filtered Channels (>1 Occurrence): {len(df_filtered_channels)}")

## 🎥 5. Fetch Top 50 Long-Form Videos per Filtered Channel & Export Dataset

For every channel passing the occurrence filter, we query `channels.list` and `playlistItems.list` on the long-form upload playlist ID (`UULF` prefix) to retrieve the most recent 50 long-form videos (>60 seconds duration). The compiled video dataset is saved directly to Google Drive (`/content/drive/MyDrive/YouTube_Analytics/`).

In [ ]:
def fetch_channel_recent_longform_videos(youtube_client, channel_id: str, channel_title: str, max_videos: int = 50):
    # Resolves long-form upload playlist (UULF ID) and fetches recent long-form videos (>60s)
    videos = []
    if not youtube_client or not channel_id:
        return videos

    try:
        # Fetch channel snippet and contentDetails
        ch_res = youtube_client.channels().list(part="snippet,contentDetails", id=channel_id).execute()
        items = ch_res.get("items", [])
        if not items:
            # Try resolving as handle if ID is not standard UC...
            handle = channel_id if channel_id.startswith("@") else "@" + channel_id
            ch_res = youtube_client.channels().list(part="snippet,contentDetails", forHandle=handle).execute()
            items = ch_res.get("items", [])
            if not items:
                return videos

        ch_item = items[0]
        uploads_pl = ch_item["contentDetails"]["relatedPlaylists"]["uploads"]
        longform_pl = "UULF" + uploads_pl[2:] if uploads_pl.startswith("UU") else uploads_pl

        next_page = None
        curr_pl = longform_pl

        while len(videos) < max_videos:
            try:
                pl_req = youtube_client.playlistItems().list(
                    part="snippet,contentDetails",
                    playlistId=curr_pl,
                    maxResults=min(50, (max_videos - len(videos)) * 2),
                    pageToken=next_page
                )
                pl_res = pl_req.execute()
            except Exception:
                if curr_pl != uploads_pl:
                    curr_pl = uploads_pl
                    next_page = None
                    continue
                else:
                    break

            pl_items = pl_res.get("items", [])
            if not pl_items:
                break

            vid_ids = [item["contentDetails"]["videoId"] for item in pl_items]
            v_req = youtube_client.videos().list(part="snippet,contentDetails,statistics", id=",".join(vid_ids))
            v_res = v_req.execute()

            for v_item in v_res.get("items", []):
                dur_iso = v_item["contentDetails"]["duration"]
                dur_sec = parse_iso8601_duration(dur_iso)
                if dur_sec <= 60:
                    continue  # Exclude Shorts

                snip = v_item["snippet"]
                stats = v_item.get("statistics", {})
                videos.append({
                    "channel_id": channel_id,
                    "channel_title": channel_title,
                    "video_id": v_item["id"],
                    "video_title": snip.get("title", ""),
                    "published_at": snip.get("publishedAt", ""),
                    "duration_seconds": dur_sec,
                    "view_count": int(stats.get("viewCount", 0)),
                    "like_count": int(stats.get("likeCount", 0)),
                    "comment_count": int(stats.get("commentCount", 0))
                })
                if len(videos) >= max_videos:
                    break

            next_page = pl_res.get("nextPageToken")
            if not next_page:
                break

    except Exception as e:
        print(f"⚠️ Error fetching videos for channel {channel_title} ({channel_id}): {e}")

    return videos

# Fetch Recent Long-Form Videos per Filtered Channel
youtube = get_youtube_client(CONFIG["YOUTUBE_API_KEY"])
all_channel_videos = []

# Check cache on Drive first for fetched channel videos dataset
videos_csv_path = os.path.join(ACTIVE_OUTPUT_DIR, f"{safe_channel_id}_subscribed_channels_recent_videos.csv")
videos_json_path = os.path.join(ACTIVE_OUTPUT_DIR, f"{safe_channel_id}_subscribed_channels_recent_videos.json")

if os.path.exists(videos_json_path) and CONFIG["USE_CACHE"]:
    print(f"⚡ Reusing cached subscribed channels video dataset from Drive: {videos_json_path}")
    with open(videos_json_path, "r", encoding="utf-8") as f:
        cache_payload = json.load(f)
    all_channel_videos = cache_payload.get("videos", [])
else:
    print(f"Starting video extraction for {len(df_filtered_channels)} filtered channels...")
    for idx, row in df_filtered_channels.iterrows():
        c_id = row["subscribed_channel_id"]
        c_title = row["subscribed_channel_title"]
        print(f" [{idx+1}/{len(df_filtered_channels)}] Fetching videos for: {c_title}")
        ch_vids = fetch_channel_recent_longform_videos(youtube, c_id, c_title, CONFIG["TARGET_VIDEOS_PER_CHANNEL"])
        all_channel_videos.extend(ch_vids)

    # Export Dataset to Drive and Local Storage
    df_all_vids = pd.DataFrame(all_channel_videos)
    if not df_all_vids.empty:
        df_all_vids.to_csv(videos_csv_path, index=False)
    export_payload = {
        "extracted_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "total_channels": len(df_filtered_channels),
        "total_videos": len(all_channel_videos),
        "videos": all_channel_videos
    }
    with open(videos_json_path, "w", encoding="utf-8") as f:
        json.dump(export_payload, f, indent=2, ensure_ascii=False)
    print(f"📁 Successfully exported channel videos dataset to Drive: {videos_json_path}")

print(f"📹 Total Channel Videos Collected: {len(all_channel_videos)}")

## 🧮 6. Video Title Vectorization, Channel Mean Embeddings & 8-Cluster K-Means

In this section, we extract video titles for each channel, compute textual embeddings, calculate the mean embedding vector per channel $v_c = \frac{1}{N_c} \sum_{i=1}^{N_c} e(t_{c,i})$, and perform **K-Means clustering** to partition the channels into 8 clusters ($K=8$).

In [ ]:
df_vids = pd.DataFrame(all_channel_videos) if all_channel_videos else pd.DataFrame()

channel_vectors = {}
channel_metadata = {}

if not df_vids.empty:
    # Group video titles by channel
    grouped = df_vids.groupby("channel_id")

    # Use TF-IDF vectorizer over video titles to build dense text feature representation
    all_titles = df_vids["video_title"].tolist()
    vectorizer = TfidfVectorizer(max_features=256, stop_words="english")
    vectorizer.fit(all_titles)

    for c_id, group in grouped:
        c_title = group["channel_title"].iloc[0]
        titles = group["video_title"].tolist()
        # Transform titles to TF-IDF embeddings and compute mean channel vector
        tfidf_matrix = vectorizer.transform(titles).toarray()
        mean_vector = np.mean(tfidf_matrix, axis=0)

        channel_vectors[c_id] = mean_vector
        channel_metadata[c_id] = {
            "channel_id": c_id,
            "channel_title": c_title,
            "video_count": len(group),
            "videos": group.sort_values(by="view_count", ascending=False).to_dict(orient="records")
        }

# Perform K-Means Clustering
channel_ids_list = list(channel_vectors.keys())
X_vectors = np.array([channel_vectors[cid] for cid in channel_ids_list]) if channel_ids_list else np.array([])

k_clusters = min(CONFIG["NUM_CLUSTERS"], len(channel_ids_list)) if len(channel_ids_list) > 0 else 0

if k_clusters > 0:
    kmeans = KMeans(n_clusters=k_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_vectors)

    # Map cluster assignments
    cluster_groups = {i: [] for i in range(k_clusters)}
    for cid, label in zip(channel_ids_list, cluster_labels):
        cluster_groups[label].append(channel_metadata[cid])
else:
    cluster_groups = {}

print(f"✅ K-Means clustering completed into {k_clusters} numerical clusters.")
for cid, chans in cluster_groups.items():
    print(f"   - Cluster {cid}: {len(chans)} channels")

## 🤖 7. Sequential LLM Cluster Synthesis (Largest to Smallest with Context Ingestion)

In accordance with project prompt instructions, we sort the numerical clusters by size (channel count, largest to smallest) and iterate through each cluster calling `gemini-3.1-flash-lite` via `google-genai` SDK. Each LLM request ingests all previously generated cluster descriptions as context to prevent generic descriptions.

In [ ]:
# Sort clusters by channel count (largest to smallest)
sorted_cluster_ids = sorted(cluster_groups.keys(), key=lambda k: len(cluster_groups[k]), reverse=True)

initial_cluster_descriptions = []
accumulated_context = []

print(f"🚀 Generating sequential LLM descriptions for {len(sorted_cluster_ids)} clusters (largest to smallest)...")

for rank_idx, cluster_id in enumerate(sorted_cluster_ids, start=1):
    chans = cluster_groups[cluster_id]
    num_channels = len(chans)

    # Format channel details and top 10 videos per channel
    chan_blocks = []
    for ch in chans:
        c_title = ch["channel_title"]
        top_10 = ch["videos"][:10]
        vid_lines = [f"    - [{v.get('view_count', 0):,} views] {v['video_title']}" for v in top_10]
        chan_blocks.append(f"  * Channel: {c_title}\n" + "\n".join(vid_lines))

    chans_text = "\n\n".join(chan_blocks)
    prev_context_text = "\n\n".join(accumulated_context) if accumulated_context else "None (First Cluster)"

    prompt = f"""You are an expert media research director and audience strategist.

You are analyzing Cluster #{rank_idx} containing {num_channels} YouTube channels subscribed to by our audience.

=== PREVIOUS CLUSTER DESCRIPTIONS (FOR CONTEXT TO PREVENT GENERIC OVERLAP) ===
{prev_context_text}

=== CURRENT CLUSTER CHANNEL DATA & TOP 10 PERFORMING VIDEOS ===
{chans_text}

Provide a distinct, non-generic qualitative description for Cluster #{rank_idx} in Markdown format with the following exact structure:
### Title: [Insert Creative & Precise Cluster Title]
**Short Description**: [2-3 sentences summarizing the niche theme, content style, and audience appeal].

**Lengthy Explanation**: [A detailed paragraph explaining why these channels belong together, the specific value proposition, and concrete examples referencing video titles and channel styles].
"""

    cache_key = f"cluster_desc_{safe_channel_id}_rank_{rank_idx}_cluster_{cluster_id}"
    try:
        desc_output = call_gemini_with_retry(prompt, cache_key=cache_key)
    except Exception as e:
        print(f"⚠️ Error generating cluster description for Rank #{rank_idx}: {e}")
        desc_output = f"### Title: Cluster #{rank_idx} ({num_channels} Channels)\n**Short Description**: Audience cluster centered around niche media topics.\n**Lengthy Explanation**: Detailed qualitative analysis unavailable due to API timeout."

    initial_cluster_descriptions.append({
        "rank": rank_idx,
        "cluster_id": cluster_id,
        "channel_count": num_channels,
        "description_text": desc_output,
        "channels": chans
    })
    accumulated_context.append(f"Cluster #{rank_idx} Summary:\n{desc_output}")
    print(f"   ✅ Generated description for Cluster Rank #{rank_idx} ({num_channels} channels)")

print("✅ All initial cluster descriptions synthesized successfully.")

## 📄 8. Final LLM Refinement & Dual-Payload Export (Markdown & JSON)

Submits a final LLM request ingesting all initial cluster descriptions to synthesize superior, refined conceptual clusters beyond strict numerical boundaries. Exports a comprehensive Markdown report (`{Channel_Name}_refined_subscription_clusters.md`) and JSON payload containing the refined cluster descriptions followed by individual cluster breakdowns (including channel titles and top 3 performing videos per channel).

In [ ]:
# Build final refinement prompt ingesting all initial cluster descriptions
all_desc_text = "\n\n---\n\n".join([f"Cluster Rank #{item['rank']} ({item['channel_count']} channels):\n{item['description_text']}" for item in initial_cluster_descriptions])

refinement_prompt = f"""You are a Chief Market Research Officer and Media Ecosystem Analyst.

Below are the 8 initial numerical cluster descriptions generated from the common subscriptions of our YouTube audience:

{all_desc_text}

Perform a final meta-synthesis to define REFINED, HIGHER-LEVEL CONCEPTUAL CLUSTERS beyond the numerical boundaries.

Provide your output in Markdown format with the following structure:
# 🌟 Refined Audience Interest Ecosystem & Strategic Cluster Synthesis

## Executive Summary of Refined Media Clusters
[Provide a high-level overview of the audience media consumption landscape].

## Refined Conceptual Clusters
1. **Refined Cluster 1: [Theme Title]**
   - **Core Essence**: [Description]
   - **Key Channels & Examples**: [Examples]
2. **Refined Cluster 2: [Theme Title]**
   - **Core Essence**: [Description]
   - **Key Channels & Examples**: [Examples]
(Continue for all major refined themes).

## Strategic Content & Partnership Implications
[Actionable audience strategy recommendations].
"""

refinement_cache_key = f"final_refinement_{safe_channel_id}"
try:
    refined_clusters_report = call_gemini_with_retry(refinement_prompt, cache_key=refinement_cache_key)
except Exception as e:
    print(f"⚠️ Error during final LLM cluster refinement: {e}")
    refined_clusters_report = f"# 🌟 Refined Audience Interest Ecosystem\n\n*Refinement report error: {e}*"

# Assemble complete Markdown document payload
md_blocks = [
    f"# 📊 Cross-Channel Subscription Clusters & Audience Affinity Report",
    f"**Target Channel**: `{CONFIG['CHANNEL_IDENTIFIER']}`",
    f"**Generated Date**: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"**Model Engine**: `{CONFIG['MODEL_NAME']}` via `google-genai` SDK",
    "\n---\n",
    refined_clusters_report,
    "\n---\n",
    "# 📌 Individual Numerical Cluster Breakdown (Sorted Largest to Smallest)\n"
]

for item in initial_cluster_descriptions:
    md_blocks.append(f"## Cluster Rank #{item['rank']} ({item['channel_count']} Channels)")
    md_blocks.append(item["description_text"])
    md_blocks.append("\n### Included Channels & Top 3 Performing Videos:")
    for ch in item["channels"]:
        md_blocks.append(f"- **Channel**: {ch['channel_title']}")
        top_3 = ch["videos"][:3]
        for v in top_3:
            md_blocks.append(f"  * [{v.get('view_count', 0):,} views] {v['video_title']}")
    md_blocks.append("\n---\n")

full_md_report = "\n".join(md_blocks)

# Export Markdown and JSON files to Drive and local fallback paths
export_md_file = f"{safe_channel_id}_refined_subscription_clusters.md"
export_json_file = f"{safe_channel_id}_refined_subscription_clusters.json"

for d in set([ACTIVE_OUTPUT_DIR] + CONFIG["LOCAL_FALLBACK_DIRS"]):
    try:
        os.makedirs(d, exist_ok=True)
        with open(os.path.join(d, export_md_file), "w", encoding="utf-8") as f:
            f.write(full_md_report)
        json_payload = {
            "channel_identifier": CONFIG["CHANNEL_IDENTIFIER"],
            "generated_at": datetime.datetime.now().isoformat(),
            "refined_synthesis": refined_clusters_report,
            "initial_clusters": initial_cluster_descriptions
        }
        with open(os.path.join(d, export_json_file), "w", encoding="utf-8") as f:
            json.dump(json_payload, f, indent=2, ensure_ascii=False)
    except Exception as e:
        print(f"⚠️ Error writing export to {d}: {e}")

print(f"💾 Successfully exported Refined Cluster Report to Drive:")
print(f"   - Markdown Report: {export_md_file}")
print(f"   - JSON Payload:    {export_json_file}")

## 📈 9. Visual Analytics & Cluster Distribution Projections

Presents summary visual charts detailing channel cluster sizes and a 2D PCA projection of the channel video title embeddings rendered inline with `plt.show()` and exported to image artifacts.

In [ ]:
if initial_cluster_descriptions:
    sns.set_theme(style="whitegrid", palette="muted")
    plt.rcParams.update({'font.size': 10, 'figure.titlesize': 14})

    # Chart 1: Cluster Channel Size Distribution (Ranked Largest to Smallest)
    plt.figure(figsize=(10, 5))
    ranks = [f"Rank #{item['rank']}" for item in initial_cluster_descriptions]
    counts = [item['channel_count'] for item in initial_cluster_descriptions]

    ax1 = sns.barplot(x=ranks, y=counts, palette="mako")
    plt.title("Channel Count per Subscription Cluster (Sorted Largest to Smallest)", fontsize=14, fontweight="bold", pad=15)
    plt.xlabel("Cluster Rank", fontweight="bold")
    plt.ylabel("Number of Subscribed Channels", fontweight="bold")

    for p in ax1.patches:
        height = p.get_height()
        if height > 0:
            ax1.annotate(f"{int(height)}",
                         (p.get_x() + p.get_width() / 2., height),
                         ha='center', va='bottom',
                         xytext=(0, 3),
                         textcoords='offset points',
                         fontsize=10, fontweight='bold')
    plt.tight_layout()
    fig1_path = os.path.join(ACTIVE_OUTPUT_DIR, f"{safe_channel_id}_cluster_sizes.png")
    plt.savefig(fig1_path, dpi=300, bbox_inches="tight")
    plt.show()

    # Chart 2: 2D PCA Projection of Channel Title Embeddings
    if len(X_vectors) > 1:
        plt.figure(figsize=(10, 6))
        pca = PCA(n_components=2)
        coords = pca.fit_transform(X_vectors)

        df_pca = pd.DataFrame({
            "PC1": coords[:, 0],
            "PC2": coords[:, 1],
            "Cluster": [f"Cluster {lbl}" for lbl in cluster_labels]
        })
        sns.scatterplot(data=df_pca, x="PC1", y="PC2", hue="Cluster", style="Cluster", s=120, palette="tab10")
        plt.title("2D PCA Scatter Plot of Channel Title Vector Embeddings", fontsize=14, fontweight="bold", pad=15)
        plt.xlabel("Principal Component 1", fontweight="bold")
        plt.ylabel("Principal Component 2", fontweight="bold")
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        fig2_path = os.path.join(ACTIVE_OUTPUT_DIR, f"{safe_channel_id}_cluster_pca.png")
        plt.savefig(fig2_path, dpi=300, bbox_inches="tight")
        plt.show()
else:
    print("ℹ️ No cluster data available to display visual analytics.")